In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gurobipy as gp
from gurobipy import Env, GRB

ACCESS_ID = "5bb59f37-8a25-4b42-ab65-018efb82fc3f"
SECRET    = "b06a6811-606c-4e97-9f1f-a3ff7ba47625"
LICENSEID = 2705567

env = Env(empty=True)
env.setParam('WLSAccessID', ACCESS_ID)
env.setParam('WLSSecret',   SECRET)
env.setParam('LicenseID',   LICENSEID)
env.start()

m = gp.Model('Childcare_MinCost_Optimization', env=env)

Set parameter Username
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2705567
Academic license 2705567 - for non-commercial use only - registered to tz___@columbia.edu


In [ ]:
print("Gurobi version:", gp.gurobi.version())

Gurobi version: (11, 0, 3)


In [ ]:
from pathlib import Path
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

# ==== 1. Data Loading & Sets====

In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/Group 13 - Project 1/Final Results')
INPUT_CSV = DATA_PATH /'Childcare Deserts FINAL.csv'
Capacity_by_facilities = DATA_PATH /'child_care_regulated.csv'
Potential_locations = DATA_PATH / 'potential_locations.csv'

INPUT_CSV = 'Childcare Deserts FINAL.csv'
Capacity_by_facilities = 'child_care_regulated.csv'
Potential_locations = 'potential_locations.csv'

In [ ]:
df_all = pd.read_csv(INPUT_CSV)
df_fac = pd.read_csv(Capacity_by_facilities)
df_loc = pd.read_csv(Potential_locations)

In [ ]:
# Standardize the zipcodes and numeric columns
df_all['zipcode'] = (df_all['zipcode'].astype(str)
                  .str.extract(r'(\d+)')[0]
                  .fillna('')
                  .str.zfill(5))

for c in ['facility_capacity', 'under5_capacity', 'total_capacity', '0-4', 'Total Kid Count']:
    if c in df_all.columns:
        df_all[c] = pd.to_numeric(df_all[c], errors='coerce').fillna(0.0)

df_loc['zipcode'] = (df_loc['zipcode'].astype(str)
                  .str.extract(r'(\d+)')[0]
                  .fillna('')
                  .str.zfill(5))

In [ ]:
# Build a facility table with unique pairs of (zipcode, facility_id)
df_fac = (
    df_fac.dropna(subset=['facility_id','latitude'])
      .assign(
          zipcode = df_fac['zipcode'].astype(str).str.extract(r'(\d+)')[0].fillna('').str.zfill(5),
          facility_id = df_fac['facility_id'].astype(str).str.strip()
      )
      .groupby(['zipcode', 'facility_id'], as_index=False)
      .agg(
          facility_capacity = ('total_capacity', 'max'),
          under5_capacity = ('under-5_capacity', 'max'),
          latitude = ('latitude', 'first'),
          longitude = ('longitude', 'first')
      )
)

In [ ]:
# Make sure these columns are string types
df_fac['zipcode'] = df_fac['zipcode'].astype(str).str.zfill(5)
df_fac['facility_id'] = df_fac['facility_id'].astype(str).str.strip()

# Drop missing IDs
# df_fac = df_fac[df_fac['facility_id'].str.lower() != 'nan']

# Drop duplicate facility ids
before = len(df_fac)
df_fac = df_fac.drop_duplicates(subset=['zipcode','facility_id'], keep='first')
# df_fac = df_fac.drop_duplicates(subset=['latitude','longitude'], keep='first') # do nothing on existing facilities that have same locations

after = len(df_fac)
print(f"Removed {before - after} duplicate facility entries; remaining: {after}")

Removed 0 duplicate facility entries; remaining: 15014


In [ ]:
missing_summary = (
    df_all.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,Unnamed: 0,0
1,zipcode,0
2,Center Exists,0
3,employment rate,0
4,average income,0
5,zipcode group 1,0
6,Demand Type,0
7,total_capacity,0
8,under-5_capacity,0
9,0-4,0


In [ ]:
missing_summary = (
    df_fac.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,zipcode,0
1,facility_id,0
2,facility_capacity,0
3,under5_capacity,0
4,latitude,0
5,longitude,0


In [ ]:
# Keep only the columns we need
need_cols = ['zipcode','Demand Type','under-5_capacity','total_capacity','0-4','Total Kid Count','Center Exists']
df_all= df_all[need_cols].copy()
# Build a table in this format: (unique (zipcode, facility_id))
df_all = (df_all.assign(zipcode = df_all['zipcode'].astype(str).str.extract(r'(\d+)')[0].fillna('').str.zfill(5),))

In [ ]:
display(df_fac)

,zipcode,facility_id,facility_capacity,under5_capacity,latitude,longitude
0,10001,229433,88,0.0,40.748836,-73.999810
1,10001,292419,79,0.0,40.749247,-74.001598
2,10001,350076,8,0.0,40.748296,-74.001263
3,10001,661697,16,0.0,40.748911,-74.001546
4,10001,827488,8,0.0,40.747845,-73.989419
...,...,...,...,...,...,...
15009,14905,702559,8,0.0,42.082112,-76.855004
15010,14905,808547,16,0.0,42.076661,-76.836433
15011,14905,821932,16,0.0,42.087195,-76.838861
15012,14905,843963,52,46.0,42.099337,-76.826906


In [ ]:
Z_all  = sorted(df_all['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes both existing and potential from data cleaning csv
Z_cand = sorted(df_loc['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes for candidate/potential locations
Z_has_fac = sorted(df_fac['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes for existing facilities

F = list(zip(df_fac['zipcode'].astype(str).str.zfill(5),
             df_fac['facility_id'].astype(str).str.strip()))
# set of (zipcode, facility_id) for existing facilities

# Map each zipcode to list of existing facilities that are located in that zipcode
from collections import defaultdict
zip_to_fac = defaultdict(list)
for z,f in F:
    zip_to_fac[z].append((z,f))


# ==== 2. Parameter Calculation ====

In [ ]:
# Existing capacity for each facility
nF_tot = {(r.zipcode, str(r.facility_id)): float(r.facility_capacity)
          for r in df_fac[['zipcode','facility_id','facility_capacity']].itertuples(index=False)}
nF_u5  = {(r.zipcode, str(r.facility_id)): float(r.under5_capacity)
          for r in df_fac[['zipcode','facility_id','under5_capacity']].itertuples(index=False)}

In [ ]:
# Current total capacity for each zipcode across all facilities in that zipcode
zip_tot_exists = df_fac.groupby('zipcode')['facility_capacity'].sum().to_dict()
zip_u5_exists  = df_fac.groupby('zipcode')['under5_capacity'].sum().to_dict()

In [ ]:
# Demand thresholds - High = 1/2, Normal = 1/3
def total_share(x):
    return 0.50 if str(x).strip().lower()=='high' else (1.0/3.0)

In [ ]:
# Under-5 needs 2/3 * (0–4 population)
needTot = {}
needU5  = {}
zip_info = df_all.groupby('zipcode', as_index=False)[['Demand Type','Total Kid Count','0-4']].first()
for r in zip_info.itertuples(index=False):
    z = r[0]
    dem_type  = r[1]
    kid_total = float(r[2])
    pop_0_4 = float(r[3])
    needTot[z] = int(np.ceil(total_share(dem_type) * kid_total))
    needU5[z] = int(np.ceil((2.0/3.0) * pop_0_4))

In [ ]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Distance calculation between 2 facilities using geodesic
from geopy.distance import geodesic

def calculate_distance(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).miles

existing_facility_locations = {}
for r in df_fac.itertuples(index=False):
    key = (r.zipcode, str(r.facility_id))
    existing_facility_locations[key] = (r.latitude, r.longitude)

potential_locations_by_zip = {}

for zipcode in Z_cand:
    zip_locations = df_loc[df_loc['zipcode'] == zipcode]
    potential_locations_by_zip[zipcode] = [
        (row.latitude, row.longitude) for row in zip_locations.itertuples(index=False)
    ]

# ==== 3. Model Construction ====

In [ ]:
m = gp.Model('Childcare_MinCost_FacilityExpand_ZipBuild',env=env)

In [ ]:
# Decision variables: Building new facilites by zipode
B_s = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Small') # +100 total, +50 U5
B_m = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Medium') # +200 total, +100 U5
B_l = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Large') # +400 total, +200 U5
U5_new = m.addVars(Z_cand, vtype=GRB.INTEGER, name='U5_From_New') # U5 share from new builds

In [ ]:
# Decision variables: Facility expansion by zipcode and facility_id
E_tot = m.addVars(F, vtype=GRB.INTEGER, name='Expand_Total') # per-facility total expansion
E_u5 = m.addVars(F, vtype=GRB.INTEGER, name='Expand_U5') # per-facility U5 expansion

In [ ]:
# Expansion capacity constraints for existing facilities
for (z,f) in F:
    n = nF_tot[(z,f)]
    m.addConstr(E_u5[(z,f)] <= E_tot[(z,f)], name=f'U5_le_Total[{z},{f}]')
    if n <= 0:
        m.addConstr(E_tot[(z,f)] == 0.0, name=f'NoExpandTot[{z},{f}]')
        m.addConstr(E_u5[(z,f)] == 0.0, name=f'NoExpandU5[{z},{f}]')
    else:
        m.addConstr(E_tot[(z,f)] <= min(0.2*n, 500.0), name=f'CapTot[{z},{f}]')
        m.addConstr(E_u5[(z,f)] <= E_tot[(z,f)], name=f'CapU5[{z},{f}]')

In [ ]:
# Coverage constraints per zip code: existing + expansion + new >= need
for z in Z_all:
    tot_exist = gp.quicksum(nF_tot[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))
    u5_exist = gp.quicksum(nF_u5[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))

    tot_exp = gp.quicksum(E_tot[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))
    u5_exp = gp.quicksum(E_u5[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))

    # only if z is part of Z_build otherwise set equal to 0
    cap_new = (100*B_s[z] + 200*B_m[z] + 400*B_l[z]) if z in Z_cand else gp.LinExpr(0.0)
    u5_from_new = (U5_new[z]) if z in Z_cand else gp.LinExpr(0.0)

    m.addConstr(tot_exist + tot_exp + cap_new >= needTot[z], name=f"TotalCover[{z}]")
    m.addConstr(u5_exist  + u5_exp  + u5_from_new >= needU5[z], name=f"U5Cover[{z}]")

    # Under 5 allocation capacity from new builds per zipcode
    if z in Z_cand:
      m.addConstr(U5_new[z] <=  50*B_s[z] + 100*B_m[z] + 200*B_l[z], name=f'U5CapFromBuild[{z}]')

In [ ]:
# Distance constraints (at least 0.06 miles between any two facilities)
# filter out the potential locations that are too close to existing facilities
valid_locs = {}
for z in Z_cand:
    valid_locs[z] = []
    if z in potential_locations_by_zip:

        # calculate the number of potential locations given distance limitation
        for potential_lat, potential_lon in potential_locations_by_zip[z]:
            valid = True
            for (zz, ff) in F:
                if zz == z:
                    existing_lat, existing_lon = existing_facility_locations[(zz, ff)]
                    distance = calculate_distance(potential_lat, potential_lon, existing_lat, existing_lon)
                    if distance < 0.06:
                        valid = False
                        break
            if valid:
                valid_locs[z].append((potential_lat, potential_lon))
# based on the updated set of potential locations, find the max number of potential loctaions where new facilities can be built


# "preliminary" opti model to address conflict pairs and figure out the max # of facilities that can be built
def max_count_exact(points, threshold=0.06):
    """
    points: list of (lat, lon)
    returns: integer, maximum number of non-conflicting points
    """
    n = len(points)
    # get all "conflict" pairs
    conflict_pairs = []
    for i in range(n):
        for j in range(i+1, n):
            if geodesic(points[i], points[j]).miles < threshold:
                conflict_pairs.append((i, j))

    m_sub = gp.Model()
    m_sub.Params.OutputFlag = 0
    x = m_sub.addVars(n, vtype=GRB.BINARY)
    for i, j in conflict_pairs:
        m_sub.addConstr(x[i] + x[j] <= 1)
    m_sub.setObjective(gp.quicksum(x[i] for i in range(n)), GRB.MAXIMIZE)
    m_sub.optimize()

    return int(m_sub.ObjVal) if m_sub.Status == GRB.OPTIMAL else None

for z in Z_cand:
    if z in potential_locations_by_zip:
        count = max_count_exact(potential_locations_by_zip[z], threshold = 0.06)
        m.addConstr(B_s[z] + B_m[z] + B_l[z] <= count, name=f'DistanceLimitation[{z}]')

In [ ]:
# Objective Function
build_cost = gp.quicksum(65000*B_s[z] + 95000*B_m[z] + 115000*B_l[z] for z in Z_cand)

# Initialize expansion cost
expand_base_cost = 0
ratio_var = {}
cost_var = {}

# Create piecewise linear cost variables for each facility
for (z,f) in F:
    n = nF_tot[(z, f)]
    if n > 0:
        # define breakpoints for expansion ratio
        x_points = [0.0, 0.1, 0.15, 0.2]
        # cost at each breakpoint
        cost_points = [0.0,(20000.0 + 200.0 * n) * 0.1, (20000.0 + 400.0 * n) * 0.15,(20000.0 + 1000.0 * n) * 0.2]
        ratio_var[(z,f)] = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, ub=0.2, name=f'Ratio_[{z},{f}]')
        m.addConstr(ratio_var[(z,f)] * n == E_tot[(z,f)], name=f'RatioDef_[{z},{f}]')

        # Create cost variable
        cost_var[(z,f)] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f'ExpandCost_[{z},{f}]')

        # Add piecewise linear constraint
        m.addGenConstrPWL(ratio_var[(z,f)], cost_var[(z,f)], x_points, cost_points, name=f'PWL_Cost_[{z},{f}]')

        expand_base_cost += cost_var[(z,f)]

u5_equip_cost = gp.quicksum(100.0*U5_new[z] for z in Z_cand) + gp.quicksum(100.0*E_u5[(z,f)] for (z,f) in F)

m.setObjective(build_cost + expand_base_cost + u5_equip_cost, GRB.MINIMIZE)

# ==== 4. Optimization ====

In [ ]:
m.Params.OutputFlag = 1
m.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 22.3.0 22D68)

CPU model: Intel(R) Core(TM) i7-1068NG7 CPU @ 2.30GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2705567 - for non-commercial use only - registered to tz___@columbia.edu
Optimize a model with 66148 rows, 68670 columns and 133641 nonzeros
Model fingerprint: 0x65aa0889
Model has 15013 general constraints
Variable types: 30026 continuous, 38644 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+02]
  Objective range  [1e+00, 1e+05]
  Bounds range     [2e-01, 2e-01]
  RHS range        [4e-01, 7e+03]
  PWLCon x range   [1e-01, 1e-01]
  PWLCon y range   [2e+03, 6e+04]
Presolve removed 65646 rows and 67555 columns
Presolve time: 4.52s
Presolved: 502 rows, 1115 columns, 2062 nonzeros
Presolved model has 194 SOS constraint(s)
Variable types: 860 continuous, 255 integer (31 binary)
Found heuristic solution: objective 2.701046e+08

Root rela

# ==== 5. Collect Results  ====

In [ ]:
# Facility level expansions
fac_rows = []

for (z,f) in F:
    fac_rows.append({
        'zipcode': z,
        'facility_id': f,
        'E_total': float(E_tot[(z,f)].X),
        'E_u5': float(E_u5[(z,f)].X),
        'Cost_ExpandBase': float(cost_var[(z,f)].X if (z,f) in cost_var else 0.0)
    })
df_fac_sol = pd.DataFrame(fac_rows)

In [ ]:
# adding results we want in the csv file
zip_rows = []
for z in Z_all:
    zip_rows.append({
        'zipcode': z,
        'Build_Small': int(round(B_s[z].X)) if z in Z_cand else 0,
        'Build_Medium': int(round(B_m[z].X)) if z in Z_cand else 0,
        'Build_Large': int(round(B_l[z].X)) if z in Z_cand else 0,
        'U5_From_New': float(U5_new[z].X) if z in Z_cand else 0,
    })
df_zip_sol = pd.DataFrame(zip_rows)

In [ ]:
def sum_by_zip(d):
    # Initialize with all the zipcodes that we want in the report
    acc = {z: 0.0 for z in Z_all}

    # Support dicts keyed by (z,f) or by z
    for k, val in d.items():
        if isinstance(k, tuple): # (z,f)
            z = k[0]
        else: # z
            z = k
        if z in acc:
            acc[z] += float(val)
    return acc

# build the per zipcode aggregates
sum_exist_tot = sum_by_zip(nF_tot)
sum_exist_u5 = sum_by_zip(nF_u5)
sum_E_tot = sum_by_zip({k: E_tot[k].X for k in F})
sum_E_u5 = sum_by_zip({k: E_u5[k].X for k in F})

# makes sure df_zip_sol lists all zipcodes
if set(df_zip_sol['zipcode']) != set(Z_all):
    df_zip_sol = (pd.DataFrame({'zipcode': sorted(Z_all)}).merge(df_zip_sol, on='zipcode', how='left').fillna(0))

df_zip_sol['Existing_Total'] = df_zip_sol['zipcode'].map(sum_exist_tot)
df_zip_sol['Existing_U5']= df_zip_sol['zipcode'].map(sum_exist_u5)
df_zip_sol['Expand_Total']= df_zip_sol['zipcode'].map(sum_E_tot)
df_zip_sol['Expand_U5']= df_zip_sol['zipcode'].map(sum_E_u5)

df_zip_sol['New_Total']= 100*df_zip_sol['Build_Small'] + 200*df_zip_sol['Build_Medium'] + 400*df_zip_sol['Build_Large']
df_zip_sol['New_U5']= df_zip_sol['U5_From_New']

df_zip_sol['Need_Total']= df_zip_sol['zipcode'].map(needTot)
df_zip_sol['Need_U5']= df_zip_sol['zipcode'].map(needU5)

# Check if we met demand and if desert has been eliminated for each zipcode

df_zip_sol['Meet_Total_Demand']= (df_zip_sol['Existing_Total'] + df_zip_sol['Expand_Total'] + df_zip_sol['New_Total'] >= df_zip_sol['Need_Total'])
df_zip_sol['Meet_U5_Demand']= (df_zip_sol['Existing_U5'] + df_zip_sol['Expand_U5'] + df_zip_sol['New_U5'] >= df_zip_sol['Need_U5'])
df_zip_sol['Desert_Eliminated']= df_zip_sol['Meet_Total_Demand'] & df_zip_sol['Meet_U5_Demand']

In [ ]:
print("\nPreview ZIP-level:")
display(df_zip_sol.head(10))
print("Preview Facility-level:")
display(df_fac_sol.head(10))


Preview ZIP-level:


,zipcode,Build_Small,Build_Medium,Build_Large,U5_From_New,Existing_Total,Existing_U5,Expand_Total,Expand_U5,New_Total,New_U5,Need_Total,Need_U5,Meet_Total_Demand,Meet_U5_Demand,Desert_Eliminated
0,10001,0,0,2,400.0,609.0,0.0,96.0,96.0,800,400.0,698,496,True,True,True
1,10005,0,0,2,323.0,39.0,0.0,0.0,0.0,800,323.0,413,323,True,True,True
2,10007,0,0,2,400.0,284.0,0.0,4.0,4.0,800,400.0,410,404,True,True,True
3,10010,0,0,5,948.0,234.0,0.0,0.0,0.0,2000,948.0,1195,948,True,True,True
4,10012,1,0,2,409.0,24.0,0.0,0.0,0.0,900,409.0,340,409,True,True,True
5,10013,0,0,4,800.0,435.0,0.0,33.0,33.0,1600,800.0,1061,833,True,True,True
6,10016,0,0,6,1199.0,823.0,0.0,0.0,0.0,2400,1199.0,1302,1199,True,True,True
7,10017,0,0,1,129.0,107.0,0.0,0.0,0.0,400,129.0,255,129,True,True,True
8,10018,0,0,1,162.0,0.0,0.0,0.0,0.0,400,162.0,268,162,True,True,True
9,10019,0,1,3,700.0,591.0,0.0,33.0,33.0,1400,700.0,1047,733,True,True,True


Preview Facility-level:


,zipcode,facility_id,E_total,E_u5,Cost_ExpandBase
0,10001,229433,14.0,14.0,10701.818182
1,10001,292419,12.0,12.0,8197.974684
2,10001,350076,1.0,1.0,2820.000000
3,10001,661697,2.0,2.0,3140.000000
4,10001,827488,1.0,1.0,2820.000000
5,10001,837329,2.0,2.0,2932.941176
6,10001,837597,13.0,13.0,9255.238095
7,10001,893683,49.0,49.0,35436.164384
8,10001,912862,2.0,2.0,2932.941176
9,10002,101512,0.0,0.0,0.000000


In [ ]:
# # Calculate the cost on building new facilities/expansion/equipments for under-5
# def calc_expansion_cost(n, expansion):
#     if expansion <= 0:
#         return 0.0
#     elif expansion <= 0.1 * n:
#         return (20000.0 + 200.0 * n) * 0.1
#     elif expansion <= 0.15 * n:
#         return (20000.0 + 400.0 * n) * 0.15
#     elif expansion <= 0.2 * n:
#         return (20000.0 + 1000.0 * n) * 0.2
# for (z, f) in F:
#     n = nF_tot[(z, f)]
#     expansion = E_tot[(z, f)].X
#     cost = calc_expansion_cost(n, expansion)
#     df_fac_sol.loc[(df_fac_sol['zipcode'] == z) & (df_fac_sol['facility_id'] == f), 'Cost_ExpandBase'] = cost


In [ ]:

df_zip_sol['Cost_Build'] = 65000*df_zip_sol['Build_Small'] + 95000*df_zip_sol['Build_Medium'] + 115000*df_zip_sol['Build_Large']
df_zip_sol['Cost_ExpandBase'] = (
    df_fac_sol.groupby('zipcode')['Cost_ExpandBase'].sum()
    .reindex(df_zip_sol['zipcode'])
    .values
)
df_zip_sol['Cost_U5_Equip'] = 100.0*(df_zip_sol['New_U5'] + df_zip_sol["Expand_U5"])

In [ ]:
if m.status == GRB.OPTIMAL:
    print(f"Optimization complete.")
    print(f"Total minimum cost = ${m.objVal:,.2f}")
else:
    print(f"Model status: {m.status} — no optimal solution found.")

Optimization complete.
Total minimum cost = $268,278,835.82


In [ ]:
import os
from pathlib import Path

In [ ]:
SAVE_DIR = Path('/content/drive/MyDrive/Group 13 - Project 1/Final Results')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
preferred_cols = ['zipcode','Build_Small','Build_Medium','Build_Large',
    'New_Total','U5_From_New','New_U5','Existing_Total','Expand_Total','Need_Total','Meet_Total_Demand',
    'Existing_U5','Expand_U5','Need_U5','Meet_U5_Demand','Desert_Eliminated',
    'Cost_Build','Cost_ExpandBase','Cost_U5_Equip','Total_Cost']

In [ ]:
cols_to_export = [c for c in preferred_cols if c in df_zip_sol.columns]
zip_out = df_zip_sol.loc[:, cols_to_export].copy()

In [ ]:
out_path = SAVE_DIR / 'optimization_result_problem2_fin.csv'
# out_path = 'optimization_result_problem2_fin.csv'
zip_out.to_csv(out_path, index=False)

print(f"ZIP-level results saved to: {out_path}")
print("Preview:")
display(zip_out.head(20))

ZIP-level results saved to: optimization_result_problem2_fin.csv
Preview:


,zipcode,Build_Small,Build_Medium,Build_Large,New_Total,U5_From_New,New_U5,Existing_Total,Expand_Total,Need_Total,Meet_Total_Demand,Existing_U5,Expand_U5,Need_U5,Meet_U5_Demand,Desert_Eliminated,Cost_Build,Cost_ExpandBase,Cost_U5_Equip
0,10001,0,0,2,800,400.0,400.0,609.0,96.0,698,True,0.0,96.0,496,True,True,230000,78237.077697,49600.0
1,10005,0,0,2,800,323.0,323.0,39.0,0.0,413,True,0.0,0.0,323,True,True,230000,0.000000,32300.0
2,10007,0,0,2,800,400.0,400.0,284.0,4.0,410,True,0.0,4.0,404,True,True,230000,1244.444444,40400.0
3,10010,0,0,5,2000,948.0,948.0,234.0,0.0,1195,True,0.0,0.0,948,True,True,575000,0.000000,94800.0
4,10012,1,0,2,900,409.0,409.0,24.0,0.0,340,True,0.0,0.0,409,True,True,295000,0.000000,40900.0
5,10013,0,0,4,1600,800.0,800.0,435.0,33.0,1061,True,0.0,33.0,833,True,True,460000,10017.288734,83300.0
6,10016,0,0,6,2400,1199.0,1199.0,823.0,0.0,1302,True,0.0,0.0,1199,True,True,690000,0.000000,119900.0
7,10017,0,0,1,400,129.0,129.0,107.0,0.0,255,True,0.0,0.0,129,True,True,115000,0.000000,12900.0
8,10018,0,0,1,400,162.0,162.0,0.0,0.0,268,True,0.0,0.0,162,True,True,115000,NaN,16200.0
9,10019,0,1,3,1400,700.0,700.0,591.0,33.0,1047,True,0.0,33.0,733,True,True,440000,10243.536774,73300.0
